# Phase 1 — FSQA Eval Question Generation
**FSQA** | fine-tune pipeline

Generates 1,000 open-ended FSQA eval questions with reference answers.
Teacher: Claude Fable 5 via OpenRouter, pinned to the AWS Bedrock BYOK
endpoint so calls bill your own AWS Bedrock key.

Weighting: 40% FSVP (21 CFR 1 Subpart L) / 40% recall (21 CFR 7, 21 CFR 810)
/ 20% PCHF (21 CFR 117)

---
## Before running — notebook setup

In the Modal Notebooks UI, attach:
1. **Secret** `your-modal-secret` — must contain `OPENROUTER_API_KEY`.
   Attached secrets are injected as environment variables automatically.
2. **Volume** `<MODAL_VOLUME_NAME>`, mounts under `./local-data`.

Also confirm your OpenRouter account has an **AWS Bedrock key** attached
under Settings → Provider Keys. Without it, `allow_fallbacks: False` makes
every call fail — which is intended (better than silently billing the wrong
pool), but you want to know why.

Run `SMOKE_TEST = True` first. Read all 40 questions. Verify the citations
are real. Then set it to False.

In [ ]:
import concurrent.futures
import itertools
import json
import os
import re
import time
from collections import Counter
from datetime import datetime, timezone

try:
    import requests
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests

## Config

In [ ]:
# --- Run mode -------------------------------------------------------------
SMOKE_TEST = True          # True = 2 batches (40 questions). False = full 50.

# --- Model ----------------------------------------------------------------
# Verified against openrouter.ai/models, Aug 2026.
# Alternative: "~anthropic/claude-fable-latest" tracks the newest Fable.
MODEL_SLUG = "anthropic/claude-fable-5"

# --- Provider pinning -----------------------------------------------------
# allow_fallbacks False means a failed BYOK call errors instead of falling
# through to OpenRouter's shared pool (which bills OpenRouter credits).
PROVIDER_CONFIG = {
    "order": ["amazon-bedrock"],
    "allow_fallbacks": False,
}

# --- Paths ----------------------------------------------------------------
# Local data directory. Put the eval sets here before running.
VOLUME_MOUNT = "./local-data"
OUTPUT_SUBDIR = "phase1_fsqa_eval"

# --- Generation -----------------------------------------------------------
QUESTIONS_PER_BATCH = 20
DIFFICULTY_SPLIT = (6, 10, 4)   # recall / application / reconciliation
MAX_WORKERS = 8
REQUEST_TIMEOUT = 300
MAX_RETRIES = 4
GEN_MAX_TOKENS = 12000     # output cap for each generation call (20 questions fit well under this); used as max_tokens

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "OPENROUTER_API_KEY not found. Attach the 'your-modal-secret' "
        "secret to this notebook and restart the kernel."
    )

if not os.path.isdir(VOLUME_MOUNT):
    raise RuntimeError(
        f"{VOLUME_MOUNT} not found. Attach the '<MODAL_VOLUME_NAME>' volume "
        f"to this notebook. Current ./local-data contents: {os.listdir('./local-data') if os.path.isdir('./local-data') else 'missing'}"
    )

# --- Spend guard -----------------------------------------------------------
# Every OpenRouter call is reserved at its worst case before it is sent and settled after (guarded_post).
# A call is refused, and the run stops, if its model has no entry in PRICE, if its body has no max_tokens,
# or if its worst case would take the run past RUN_BUDGET_USD. PRICE holds each model's listed price in
# USD per million tokens (input, output). Entries the repo does not state are left for the operator to fill.
from datetime import datetime, timezone
import math


# RUN_BUDGET_USD: unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_positive_float(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _ledger_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


RUN_BUDGET_USD = _env_positive_float("RUN_BUDGET_USD", 25.00)   # one cap per run; override with the RUN_BUDGET_USD environment variable
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
# Confirmed listed price, USD per million tokens (input, output), highest endpoint price found.
# Source: https://openrouter.ai/api/v1/models/anthropic/claude-fable-5/endpoints, retrieved 2026-11-05.
# The Vertex Europe regional endpoint lists 11 / 55; Claude Platform on AWS, Azure, Bedrock, Vertex global and Anthropic list 10 / 50.
# Add a model here only with a confirmed price. A model with no entry makes the notebook refuse to run.
PRICE = {"anthropic/claude-fable-5": (11, 55)}


def _valid_price(p):
    return (isinstance(p, (tuple, list)) and len(p) == 2
            and all(isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x) and x > 0 for x in p))


missing_prices = sorted(m for m in {MODEL_SLUG} if not _valid_price(PRICE.get(m)))
if missing_prices:
    raise RuntimeError(f"Refusing to run: no confirmed listed price for {missing_prices}. Set (input $/M, output $/M) for each "
                       "in the PRICE table in this config cell, then rerun. No paid call was made.")
# None is the only signal for a new run. Any other value is a resume ID and goes through the two checks below,
# which refuse an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if not isinstance(RUN_ID, str) or not RUN_ID.strip():
    raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {RUN_ID!r}).")
if not re.fullmatch(r"[A-Za-z0-9_.-]+", RUN_ID) or RUN_ID in (".", ".."):
    raise RuntimeError(f"RUN_ID {RUN_ID!r} must use only letters, digits, underscore, dot and hyphen.")
RUN_DIR = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
# One ledger and lock root for every notebook, from the canonical rule in _ledger_root() above.
SPEND_LEDGER_DIR = _ledger_root()
LEDGER_DIR = os.path.join(SPEND_LEDGER_DIR, RUN_ID)
os.makedirs(LEDGER_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(LEDGER_DIR, "spend_ledger.jsonl")

# One paid runner at a time across every notebook. An exclusive lock file at SPEND_LEDGER_DIR/runner.lock: a second
# runner refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died without
# exiting) is never cleared automatically: confirm no runner is alive, then delete the lock file by hand to resume.
import atexit
RUNNER_LOCK = os.path.join(SPEND_LEDGER_DIR, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another paid run may be active, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


print("API key:      loaded")
print(f"Volume mount: {VOLUME_MOUNT}  OK")
print(f"Model:        {MODEL_SLUG}")
print(f"Provider:     {PROVIDER_CONFIG}")
print(f"Budget cap:   ${RUN_BUDGET_USD:.2f} per run (RUN_BUDGET_USD)")
print(f"Ledger:       {LEDGER_PATH}")
print(f"Mode:         {'SMOKE TEST (2 batches)' if SMOKE_TEST else 'FULL RUN (50 batches)'}")

## Seed plan — CFR sections × question archetypes

One seed per batch. No section/archetype pair repeats, which is what keeps
batch 40 from looking like batch 4.

In [ ]:
ARCHETYPES = [
    "applicability",
    "timeline or deadline",
    "recordkeeping requirement",
    "exemption or qualified exemption",
    "definition",
    "procedural sequence",
    "numeric threshold",
    "responsible party",
    "consequence of non-compliance",
    "conflict between two provisions",
]

FSVP_SECTIONS = [
    "21 CFR 1.500 — Definitions",
    "21 CFR 1.501 — To whom does this subpart apply",
    "21 CFR 1.502 — What FSVP must I have",
    "21 CFR 1.503 — Qualified individual requirements",
    "21 CFR 1.504 — Hazard analysis",
    "21 CFR 1.505 — Evaluation for foreign supplier approval and verification",
    "21 CFR 1.506 — Foreign supplier verification activities",
    "21 CFR 1.507 — Verification activities for certain FSVP importers",
    "21 CFR 1.508 — Written procedures for ensuring receipt from approved suppliers",
    "21 CFR 1.509 — Corrective actions",
    "21 CFR 1.510 — Records",
    "21 CFR 1.511 — Modified requirements — very small importers and small foreign suppliers",
    "21 CFR 1.512 — Modified requirements — dietary supplements",
    "21 CFR 1.513 — Exemptions",
    "21 CFR 1.514 — Consequences of failing to comply",
]

RECALL_SECTIONS = [
    "21 CFR 7.40 — Recall policy",
    "21 CFR 7.41 — Health hazard evaluation and recall classification",
    "21 CFR 7.42 — Recall strategy",
    "21 CFR 7.45 — FDA-requested recall",
    "21 CFR 7.46 — Firm-initiated recall",
    "21 CFR 7.49 — Recall communications",
    "21 CFR 7.50 — Public notification of recall",
    "21 CFR 7.53 — Recall status reports",
    "21 CFR 7.55 — Termination of a recall",
    "21 CFR 7.59 — General industry guidance",
    "21 CFR 810.10 — Cease distribution and notification order",
    "21 CFR 810.11 — Regulation review",
    "21 CFR 810.12 — Mandatory recall order",
    "21 CFR 810.15 — Recall strategy",
    "21 CFR 810.16 — Communications concerning a mandatory recall order",
    "21 CFR 810.17 — Recall status reports",
    "21 CFR 810.18 — Termination of a mandatory recall order",
]

PCHF_SECTIONS = [
    "21 CFR 117.3 — Definitions",
    "21 CFR 117.5 — Exemptions",
    "21 CFR 117.126 — Food safety plan",
    "21 CFR 117.130 — Hazard analysis",
    "21 CFR 117.135 — Preventive controls",
    "21 CFR 117.139 — Recall plan",
    "21 CFR 117.140 — Preventive control management components",
    "21 CFR 117.150 — Corrective actions and corrections",
    "21 CFR 117.155 — Verification",
    "21 CFR 117.160 — Validation",
    "21 CFR 117.165 — Verification of implementation and effectiveness",
    "21 CFR 117.170 — Reanalysis",
    "21 CFR 117.180 — Preventive controls qualified individual",
    "21 CFR 117.190 — Implementation records",
]

BATCH_ALLOCATION = [
    ("fsvp", FSVP_SECTIONS, 20),
    ("recall", RECALL_SECTIONS, 20),
    ("pchf", PCHF_SECTIONS, 10),
]


def build_seeds():
    seeds = []
    for domain, sections, n_batches in BATCH_ALLOCATION:
        sec_cycle = itertools.cycle(sections)
        arch_cycle = itertools.cycle(ARCHETYPES)
        seen = set()
        while len([s for s in seeds if s["domain"] == domain]) < n_batches:
            section, archetype = next(sec_cycle), next(arch_cycle)
            key = (section, archetype)
            if key in seen:
                continue
            seen.add(key)
            seeds.append({
                "batch_id": f"{domain}-{len(seen):03d}",
                "domain": domain,
                "section": section,
                "archetype": archetype,
            })
    return seeds


SEEDS = build_seeds()
if SMOKE_TEST:
    # One FSVP, one recall — checks both prompt shapes
    SEEDS = [SEEDS[0], SEEDS[20]]

print(f"Seeds: {len(SEEDS)}")
print(f"By domain: {dict(Counter(s['domain'] for s in SEEDS))}")
print(f"Questions to generate: {len(SEEDS) * QUESTIONS_PER_BATCH}")

In [ ]:
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if not _valid_price(PRICE.get(model)):
        raise BudgetExceeded(f"not sent: no confirmed price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if not _valid_price(PRICE.get(model)):
        raise BudgetExceeded(f"not sent: no confirmed price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int) or isinstance(body.get("max_tokens"), bool) or body["max_tokens"] <= 0:
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def guarded_post(body, headers, timeout=REQUEST_TIMEOUT, purpose="call"):
    """One POST to OpenRouter. The attempt is reserved at its worst case before it is sent and settled after, so
    retries, exceptions and rejected calls are all counted. BudgetExceeded is raised before anything is sent."""
    body = openrouter_body(body)
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = requests.post(OPENROUTER_URL, headers=headers, json=body, timeout=timeout)
        if r.status_code == 200:
            try:
                usage = (r.json() or {}).get("usage") or None
            except Exception:
                usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


import json, requests, os

def probe(label, body):
    r = guarded_post(body, {"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
                 "Content-Type": "application/json"}, timeout=120, purpose="probe")
    print(f"\n=== {label} ===")
    print(f"HTTP {r.status_code}")
    print(r.text[:1200])
    return r

msg = [{"role": "user", "content": "Reply with the word OK."}]

# A: bare call — does the slug work at all?
probe("A: bare", {"model": MODEL_SLUG, "messages": msg, "max_tokens": 20})

# B: + provider pin — is BYOK/Bedrock the problem?
probe("B: provider pin", {"model": MODEL_SLUG, "messages": msg, "max_tokens": 20,
                          "provider": PROVIDER_CONFIG})

# C: + structured output — does this endpoint support json_schema?
probe("C: response_format", {"model": MODEL_SLUG, "messages": msg, "max_tokens": 200,
                             "response_format": RESPONSE_SCHEMA})

## Prompt and output schema

In [ ]:
SYSTEM_PROMPT = (
    "You are building an evaluation set for a food safety regulatory model. "
    "The questions will be used to score several LLMs on FSQA reasoning, then "
    "the winner generates training data for a fine-tune. Bad questions produce "
    "a bad model, so precision matters more than volume."
)

USER_TEMPLATE = """Generate {n} questions from this seed:

CFR SECTION: {section}
ARCHETYPE: {archetype}

Each question must:
- Be answerable from the named CFR section alone
- Be open-ended. No multiple choice, no yes/no
- Describe a real situation a QA manager or FSVP importer would face
- Use the words practitioners use, not regulatory boilerplate
- Be distinct from the other {n_minus_1} in this batch

Each reference answer must:
- Cite the specific subsection, not just the part number
- State the requirement plainly in under 120 words
- Note any condition or exception that changes the answer
- Contain nothing that isn't in the cited text

Difficulty mix for this batch: {n_recall} straightforward recall, {n_app} requiring application to the scenario, {n_rec} requiring reconciliation of two or more provisions.

If the seed section does not support {n} distinct questions of this archetype, return fewer. Do not pad. Do not invent subsections that do not exist; if you are unsure a citation is real, omit that question.

Return JSON only, matching the schema."""

RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "fsqa_eval_batch",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "questions": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "id": {"type": "string"},
                            "question": {"type": "string"},
                            "reference_answer": {"type": "string"},
                            "citation": {"type": "string"},
                            "archetype": {"type": "string"},
                            "difficulty": {
                                "type": "string",
                                "enum": ["recall", "application", "reconciliation"],
                            },
                        },
                        "required": ["id", "question", "reference_answer",
                                     "citation", "archetype", "difficulty"],
                        "additionalProperties": False,
                    },
                }
            },
            "required": ["questions"],
            "additionalProperties": False,
        },
    },
}

# Format check only. Confirms a cite is SHAPED like a real one; does not
# confirm the subsection exists. Hallucinated subsections are the failure
# mode this cannot catch — that's what the smoke test read-through is for.
CITATION_PATTERN = re.compile(r"21\s*CFR\s*(1\.\d{3}|7\.\d{2}|810\.\d{2}|117\.\d{1,3})")

## Generation

In [ ]:
HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "X-OpenRouter-Title": "labs-benchmark",
}


def generate_batch(seed: dict) -> dict:
    """One batch of questions. Returns a result envelope; never raises."""
    n = QUESTIONS_PER_BATCH
    n_recall, n_app, n_rec = DIFFICULTY_SPLIT

    payload = {
        "model": MODEL_SLUG,
        "provider": PROVIDER_CONFIG,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(
                n=n, n_minus_1=n - 1,
                section=seed["section"], archetype=seed["archetype"],
                n_recall=n_recall, n_app=n_app, n_rec=n_rec,
            )},
        ],
        "response_format": RESPONSE_SCHEMA,
        "max_tokens": GEN_MAX_TOKENS,
    }

    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            r = guarded_post(payload, HEADERS, timeout=REQUEST_TIMEOUT, purpose="gen")

            if r.status_code in (429, 503):
                wait = r.headers.get("Retry-After")
                delay = float(wait) if wait and wait.isdigit() else 2 ** (attempt + 2)
                print(f"  [{seed['batch_id']}] {r.status_code} — waiting {delay}s")
                time.sleep(delay)
                last_error = f"HTTP {r.status_code}"
                continue

            if r.status_code >= 500:
                time.sleep(2 ** attempt)
                last_error = f"HTTP {r.status_code}"
                continue

            r.raise_for_status()
            data = r.json()

            # OpenRouter can return errors inside a 200
            if "error" in data:
                last_error = str(data["error"])[:300]
                print(f"  [{seed['batch_id']}] API error: {last_error}")
                time.sleep(2 ** attempt)
                continue

            content = data["choices"][0]["message"]["content"]
            parsed = json.loads(
                content.replace("```json", "").replace("```", "").strip()
            )
            questions = parsed.get("questions", [])

            clean, rejected = [], []
            for i, q in enumerate(questions):
                q.update({
                    "id": f"{seed['batch_id']}-{i+1:02d}",
                    "batch_id": seed["batch_id"],
                    "domain": seed["domain"],
                    "seed_section": seed["section"],
                    "seed_archetype": seed["archetype"],
                })
                if not CITATION_PATTERN.search(q.get("citation", "")):
                    q["_reject_reason"] = "citation failed format check"
                    rejected.append(q)
                elif len(q.get("question", "")) < 20:
                    q["_reject_reason"] = "question too short"
                    rejected.append(q)
                else:
                    clean.append(q)

            print(f"  [{seed['batch_id']}] {len(clean)} kept, {len(rejected)} rejected")
            return {
                "batch_id": seed["batch_id"], "status": "success",
                "questions": clean, "rejected": rejected,
                "returned": len(questions), "kept": len(clean),
                "provider": data.get("provider"),
                "usage": data.get("usage", {}), "error": None,
            }

        except BudgetExceeded:
            raise
        except Exception as e:
            last_error = f"{type(e).__name__}: {str(e)[:300]}"
            print(f"  [{seed['batch_id']}] attempt {attempt+1} failed: {last_error}")
            time.sleep(2 ** attempt)

    print(f"  [{seed['batch_id']}] FAILED after {MAX_RETRIES} attempts")
    return {
        "batch_id": seed["batch_id"], "status": "failed",
        "questions": [], "rejected": [], "returned": 0, "kept": 0,
        "provider": None, "usage": {}, "error": last_error,
    }

In [ ]:
run_id = RUN_ID
print(f"Run {run_id} — {len(SEEDS)} batches, {MAX_WORKERS} workers")
print("-" * 62)

start = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    results = list(pool.map(generate_batch, SEEDS))
elapsed = time.time() - start

print("-" * 62)
print(f"Done in {elapsed:.0f}s")

## Write to volume

In [ ]:
outdir = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, run_id)
os.makedirs(outdir, exist_ok=True)

all_q, all_rej, failed = [], [], []
tok_in = tok_out = 0

for res in results:
    all_q.extend(res["questions"])
    all_rej.extend(res["rejected"])
    if res["status"] != "success":
        failed.append({"batch_id": res["batch_id"], "error": res["error"]})
    u = res.get("usage") or {}
    tok_in += u.get("prompt_tokens", 0)
    tok_out += u.get("completion_tokens", 0)

q_path = os.path.join(outdir, "questions.jsonl")
with open(q_path, "w") as f:
    for q in all_q:
        f.write(json.dumps(q) + "\n")

rej_path = os.path.join(outdir, "rejected.jsonl")
with open(rej_path, "w") as f:
    for q in all_rej:
        f.write(json.dumps(q) + "\n")

providers_used = sorted({r["provider"] for r in results if r.get("provider")})

manifest = {
    "run_id": run_id,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "smoke_test": SMOKE_TEST,
    "model": MODEL_SLUG,
    "provider_config": PROVIDER_CONFIG,
    "providers_used": providers_used,
    "elapsed_seconds": round(elapsed, 1),
    "batches_attempted": len(results),
    "batches_failed": len(failed),
    "failures": failed,
    "questions_kept": len(all_q),
    "questions_rejected": len(all_rej),
    "by_domain": dict(Counter(q["domain"] for q in all_q)),
    "by_difficulty": dict(Counter(q.get("difficulty", "unknown") for q in all_q)),
    "tokens": {"prompt": tok_in, "completion": tok_out, "total": tok_in + tok_out},
    "est_cost_usd": round(tok_in / 1e6 * 10 + tok_out / 1e6 * 50, 2),
    "files": {"questions": q_path, "rejected": rej_path},
}

with open(os.path.join(outdir, "manifest.json"), "w") as f:
    json.dump(manifest, f, indent=2)

print(f"Kept:       {manifest['questions_kept']}")
print(f"Rejected:   {manifest['questions_rejected']}")
print(f"Failed:     {manifest['batches_failed']} batches")
print(f"By domain:  {manifest['by_domain']}")
print(f"Difficulty: {manifest['by_difficulty']}")
print(f"Tokens:     {manifest['tokens']['total']:,}")
print(f"Est. cost:  ${manifest['est_cost_usd']}")
print(f"Providers:  {providers_used}")
print(f"Output:     {q_path}")

if providers_used and providers_used != ["amazon-bedrock"]:
    print("\n*** WARNING ***")
    print("A provider other than amazon-bedrock served part of this run.")
    print("Check BYOK config; you may be billing the wrong account.")

## Smoke test review

Read every question below before setting `SMOKE_TEST = False`.
You are checking one thing the code cannot: **do these subsections exist,
and does the reference answer actually match what the CFR says?**

In [ ]:
if SMOKE_TEST:
    for q in all_q:
        print(f"\n[{q['id']}] {q['difficulty']} — {q['citation']}")
        print(f"Q: {q['question']}")
        print(f"A: {q['reference_answer']}")
    print(f"\n{'='*62}")
    print(f"{len(all_q)} questions above. Verify citations against eCFR.")
    print("Then set SMOKE_TEST = False and re-run.")

## Reload a previous run

In [ ]:
def load_run(run_id_to_load: str):
    path = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR, run_id_to_load, "questions.jsonl")
    with open(path) as f:
        return [json.loads(line) for line in f]


def list_runs():
    base = os.path.join(VOLUME_MOUNT, OUTPUT_SUBDIR)
    return sorted(os.listdir(base)) if os.path.isdir(base) else []


print("Available runs:", list_runs())